<a href="https://colab.research.google.com/github/Shaimaa307/flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Shaimaa307/flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

Finding 1

Finding

Refreshed pages may recover search impressions.

Methodology Question

Could seasonality or topic popularity explain part of the observed improvement?

Finding 2

Finding

Pages with high impressions and low CTR may benefit from CTR improvements.

Methodology Question

Could ranking position explain the low CTR instead of page quality?

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
paper_findings = {
    "Finding 1": {
        "Finding": "Refreshed pages may recover search impressions.",
        "Question": "Could seasonality or topic popularity explain part of the observed improvement?"
    },
    "Finding 2": {
        "Finding": "Pages with high impressions and low CTR may benefit from CTR improvements.",
        "Question": "Could ranking position explain the low CTR instead of page quality?"
    }
}

for key, value in paper_findings.items():
    print(f"{key}")
    print("Finding:", value["Finding"])
    print("Methodology Question:", value["Question"])
    print()


Finding 1
Finding: Refreshed pages may recover search impressions.
Methodology Question: Could seasonality or topic popularity explain part of the observed improvement?

Finding 2
Finding: Pages with high impressions and low CTR may benefit from CTR improvements.
Methodology Question: Could ranking position explain the low CTR instead of page quality?



## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

## Honest split

In Week 5, the model used a random train/test split.

In this notebook, the model is evaluated using a grouped split by client. This reduces the chance of information leakage between the training and testing sets.

The grouped split provides a more realistic estimate of model performance.

In [9]:
!pip -q install duckdb huggingface_hub scikit-learn

from google.colab import userdata
import duckdb
import pandas as pd

HF_TOKEN = userdata.get("HF_TOKEN")

con = duckdb.connect()

con.execute(f"""
CREATE OR REPLACE SECRET hf_secret (
TYPE HUGGINGFACE,
TOKEN '{HF_TOKEN}'
);
""")

warehouse = "hf://datasets/FlyRank/internship-warehouse"

query = f"""
SELECT *
FROM read_parquet(
'{warehouse}/fact_content_daily_performance/month=2026-03/*.parquet'
)
LIMIT 100000
"""

df = con.sql(query).df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [10]:
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score
import pandas as pd
import numpy as np

# Fill missing values
cols = [
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
    "ga4_pageviews",
    "ga4_sessions",
    "sessions_organic",
    "scroll_events"
]

for c in cols:
    df[c] = df[c].fillna(0)

# CTR
df["ctr"] = np.where(
    df["gsc_impressions"] > 0,
    df["gsc_clicks"] / df["gsc_impressions"],
    0
)

# Target
df["target"] = (df["ctr"] < 0.05).astype(int)

# Features
features = [
    "gsc_avg_position",
    "ga4_pageviews",
    "ga4_sessions",
    "sessions_organic",
    "scroll_events"
]

X = df[features]
y = df["target"]

# -----------------------------
# Random Split (Week 5)
# -----------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

random_model = DecisionTreeClassifier(
    max_depth=5,
    random_state=42
)

random_model.fit(X_train, y_train)

random_pred = random_model.predict(X_test)

random_acc = accuracy_score(y_test, random_pred)

# -----------------------------
# Grouped Split
# -----------------------------
groups = df["client_hash_id"]

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.2,
    random_state=42
)

train_idx, test_idx = next(gss.split(X, y, groups))

X_train = X.iloc[train_idx]
X_test = X.iloc[test_idx]

y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]

group_model = DecisionTreeClassifier(
    max_depth=5,
    random_state=42
)

group_model.fit(X_train, y_train)

group_pred = group_model.predict(X_test)

group_acc = accuracy_score(y_test, group_pred)

comparison = pd.DataFrame({
    "Validation Method": [
        "Random Split",
        "Grouped Split"
    ],
    "Accuracy": [
        random_acc,
        group_acc
    ]
})

comparison

,Validation Method,Accuracy
0,Random Split,0.996300
1,Grouped Split,0.996816


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

## Leakage audit

The final model uses only features that are available before prediction.

Excluded fields include client identifiers and content identifiers.

No future information, product flags, or label-derived features are used.

The grouped split further reduces the risk of information leakage between training and testing.

In [11]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print("Features used:")
print(features)

print("\nExcluded columns:")
print([
    "client_hash_id",
    "content_hash_id"
])

print("\nLeakage audit")

print("- No future windows used")
print("- No product flags used")
print("- No client or content identifiers used as features")
print("- Grouped split used to reduce leakage")

Features used:
['gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'sessions_organic', 'scroll_events']

Excluded columns:
['client_hash_id', 'content_hash_id']

Leakage audit
- No future windows used
- No product flags used
- No client or content identifiers used as features
- Grouped split used to reduce leakage


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

## Original claim

The Decision Tree predicts which pages need action.

## Revised claim

In this dataset, the Decision Tree achieved higher accuracy than the baseline model.

This result is observational and should be used as decision support rather than evidence of causation.

In [12]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print("Original claim reviewed.")
print("Claim rewritten using careful research language.")

Original claim reviewed.
Claim rewritten using careful research language.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.